In [2]:
!nvidia-smi -L
!cd /home/jupyter/project/slm-audio-evidence && git status -sb && git branch -a

hi


In [ ]:
%%bash

REPO_DIR=/home/jupyter/project/slm-audio-evidence-polina

if [ ! -d "$REPO_DIR/.git" ]; then
    git clone https://github.com/PolinaSh-main/slm-audio-evidence-judge-fork.git "$REPO_DIR"
fi

cd "$REPO_DIR"
git checkout m3/llm-judge
git pull origin m3/llm-judge

In [ ]:
%pip install -q -r /home/jupyter/project/slm-audio-evidence-polina/requirements-judge.txt

In [ ]:
from pathlib import Path
FILESTORE = Path("/home/jupyter/filestore/space1/slm-audio-evidence-polina")
MODEL_CACHE = FILESTORE / "models" / "qwen3-8b"

if MODEL_CACHE.exists() and any(MODEL_CACHE.glob("*.safetensors")):
    print("Уже скачано:", MODEL_CACHE)
else:
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="Qwen/Qwen3-8B", local_dir=str(MODEL_CACHE), local_dir_use_symlinks=False)

**Заглушка `torchaudio`.** Судье звук не нужен, но `transformers` при загрузке ЛЮБОЙ модели безусловно тянет `torchaudio` (`transformers/loss/loss_rnnt.py: from .loss_rnnt import ParakeetForRNNTLoss` — только ради ASR-модели Parakeet, которую мы не используем). Системный `torchaudio` в этом DataSphere-образе не грузится ни с одной версией `torch`, которую мы пробовали (`undefined symbol` в `libtorchaudio.so` — судя по всему, несовместимая по ABI сборка конкретно в этом образе, не вопрос версии). Вместо погони за совместимой сборкой — подсовываем пустой модуль раньше настоящего пакета в `sys.path`, так реальный `.so` вообще не тронется.

In [ ]:
import os
import pathlib
import sys

_STUB_DIR = "/tmp/torchaudio_stub"
os.makedirs(_STUB_DIR, exist_ok=True)
pathlib.Path(f"{_STUB_DIR}/torchaudio.py").write_text(
    "# Заглушка -- см. markdown выше. Судье не нужны реальные функции torchaudio.\n"
)

sys.path.insert(0, _STUB_DIR)  # для импортов в этом же кернеле (ячейки ниже)
os.environ["PYTHONPATH"] = _STUB_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")  # для !-подпроцессов

import torchaudio  # noqa: E402 -- должна взяться заглушка, не сломанный системный пакет

print("torchaudio ->", torchaudio.__file__)
assert _STUB_DIR in torchaudio.__file__, "заглушка не перехватила импорт -- проверь sys.path"

In [ ]:
import json, os
os.chdir("/home/jupyter/project/slm-audio-evidence-polina")

resp_path = "results/cascade_plain_20260712/responses.jsonl"  # любой уже готовый прогон
manifest = {json.loads(l)["id"]: json.loads(l) for l in open("data/manifests/pilot.jsonl", encoding="utf-8")}
responses = [json.loads(l) for l in open(resp_path, encoding="utf-8")]
b_ids = {i for i, m in manifest.items() if m["category"] == "B"}
smoke = [r for r in responses if r["id"] in b_ids][:5]

os.makedirs("/tmp/smoke", exist_ok=True)
with open("/tmp/smoke/responses.jsonl", "w", encoding="utf-8") as f:
    for r in smoke:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"{len(smoke)} B-ответов для smoke-теста")

In [ ]:
!cd /home/jupyter/project/slm-audio-evidence-polina && python3 -m src.run_eval \
  --manifest data/manifests/pilot.jsonl \
  --responses /tmp/smoke/responses.jsonl \
  --out /tmp/smoke_out \
  --judge local \
  --judge-model /home/jupyter/filestore/space1/slm-audio-evidence-polina/models/qwen3-8b \
  --judge-display-name "Qwen/Qwen3-8B" \
  --judge-prompt judge_v1.txt

In [ ]:
print(open("/tmp/smoke_out/responses_judged.jsonl").read())

## Прогон скейл-сета (Задача 2, Шаг 2 — ROLE_M4)

Смоук-тест выше проверил только плётку (репо → зависимости → модель → судья) на 5 B-ответах из пилота. Здесь — реальный прогон замороженным судьёй (`Qwen/Qwen3-8B`, `judge_v1.txt`, no-think — 94% (87/93) на пилоте, `docs/decisions.md` 2026-07-15/16) по всем 4 прогонам скейл-сета от M2.

Судья грузится **один раз** и переиспользуется на все 4 прогона (`run_evaluation()` из `src/run_eval.py` принимает уже собранный `judge`) — та же схема, что в `notebooks/kaggle_run.ipynb`'s `run_candidate()`, просто без лестницы кандидатов: тут только один, уже зафиксированный судья, не эксперимент.

Манифест — `data/manifests/scale_nmsqa_final.jsonl` (376 позиций, заморожен 26.07 после ручного ASR-ревью), не черновой `scale_nmsqa.jsonl`. `responses.jsonl` в каждом прогоне — 380 строк; несовпадающие с финальным манифестом id `run_evaluation()` пропустит сам с предупреждением, это ожидаемо.

Результат — `responses_judged.jsonl` + `metrics.md` прямо в папке каждого прогона (`results/scale_nmsqa/<run_id>/`), рядом с уже лежащим там `responses.jsonl`.

In [ ]:
from src.judges import build_judge
from src.run_eval import run_evaluation

SCALE_MANIFEST = "data/manifests/scale_nmsqa_final.jsonl"  # заморожен 26.07 после ручного ASR-ревью
SCALE_RUNS = [
    "cascade_plain_20260725",
    "cascade_s1_idk_20260725",
    "qwen2audio_plain_20260725",
    "qwen2audio_s1_idk_20260725",
]

# Тот же замороженный конфиг, что в смоук-тесте выше -- не эксперимент, см. markdown.
scale_judge = build_judge(
    "local",
    model_id=str(MODEL_CACHE),
    display_name="Qwen/Qwen3-8B",
    prompt_name="judge_v1.txt",
    enable_thinking=False,
)

for run_id in SCALE_RUNS:
    print(f"=== {run_id} ===")
    run_evaluation(
        manifest_path=SCALE_MANIFEST,
        responses_path=f"results/scale_nmsqa/{run_id}/responses.jsonl",
        out_dir=f"results/scale_nmsqa/{run_id}",
        judge=scale_judge,
    )

In [ ]:
# Быстрая сверка: метрики каждого прогона глазами, прежде чем коммитить results/.
for run_id in SCALE_RUNS:
    print(f"\n=== {run_id} ===")
    print(open(f"results/scale_nmsqa/{run_id}/metrics.md", encoding="utf-8").read())

## Поиск `scale_nmsqa_final_dataset.zip`

Прислан в Телеграм, но сама DataSphere-сессия не видит содержимое чужого локального чата — если Telegram открыт на твоей машине (не внутри DataSphere), файл нужно сначала загрузить в DataSphere вручную (перетащить в файловый браузер JupyterLab слева, или через раздел загрузки датасетов DataSphere) — ноутбук не может дотянуться до диска твоего компьютера напрямую.

Ниже — поиск по двум персистентным точкам (`/home`, где живут и `REPO_DIR`, и `FILESTORE`, и `/tmp`) на случай, если файл уже куда-то залился, но не там, где ты его ждала. Для справки: `notebooks/colab_run.ipynb` (общий DataSphere-ноутбук M2) ожидает такие ZIP'ы в `/home/jupyter/filestore/space1/slm-audio-evidence/input/` — но это общее пространство команды, а не твоя личная `-polina` папка, так что стоит проверить обе.

In [ ]:
import subprocess

print("=== Точное совпадение по имени (scale_nmsqa_final*) ===")
exact = subprocess.run(
    ["find", "/home", "/tmp", "-iname", "*scale_nmsqa_final*"],
    capture_output=True, text=True,
).stdout.strip()
print(exact or "(не найдено)")

print("\n=== Все .zip в /home и /tmp, новые сверху (вдруг переименован) ===")
zips = subprocess.run(
    ["find", "/home", "/tmp", "-iname", "*.zip", "-printf", "%T@ %TY-%Tm-%Td %TH:%TM  %p\\n"],
    capture_output=True, text=True,
).stdout.strip().splitlines()
zips.sort(reverse=True)  # свежие сверху -- сортировка по ведущему unix-времени
print("\n".join(line.split(" ", 1)[1] for line in zips) if zips else "(zip-файлов не найдено вообще)")

## Сверка: тот же фриз или другой?

Найденная папка (`/home/jupyter/project/scale_nmsqa_final_dataset/`) содержит только манифест + ASR-ревью JSON/CSV — ровно тот же набор файлов, что `git log` показывает в коммите `7bb1a93` ("Freeze final NMSQA scale set and review scripts", 26.07) уже в репозитории под `data/`. Audio в папке нет вообще — судье он и не нужен (читает только текст из манифеста), но это также означает, что не с чем сверять "тот ли это датасет" кроме самого манифеста.

Сравниваем побайтово, прежде чем тратить GPU-время на пересчёт: если хэши совпадают — уже посчитанные `results/scale_nmsqa/*/responses_judged.jsonl` актуальны, повторный прогон не нужен.

In [ ]:
import hashlib

REPO_MANIFEST = "data/manifests/scale_nmsqa_final.jsonl"  # уже в репо, уже посчитан
TELEGRAM_MANIFEST = "/home/jupyter/project/scale_nmsqa_final_dataset/manifests/scale_nmsqa_final.jsonl"


def sha256(path: str) -> str:
    return hashlib.sha256(open(path, "rb").read()).hexdigest()


h_repo, h_telegram = sha256(REPO_MANIFEST), sha256(TELEGRAM_MANIFEST)
print("В репо:    ", h_repo, REPO_MANIFEST)
print("Из Telegram:", h_telegram, TELEGRAM_MANIFEST)

if h_repo == h_telegram:
    print("\nИДЕНТИЧНЫ -- это тот же фриз, results/scale_nmsqa/*/responses_judged.jsonl уже актуальны, "
          "повторный прогон НЕ нужен.")
else:
    print("\nРАЗЛИЧАЮТСЯ -- см. следующую ячейку для повторного прогона на файле из Telegram.")

## Повторный прогон на манифесте из Telegram (только если хэши выше различались)

Пишет в те же папки `results/scale_nmsqa/<run_id>/`, что и первый прогон — перезатирает `responses_judged.jsonl`/`metrics.md`/`judge_cache.jsonl` версией на актуальном манифесте. Если `scale_judge`/`MODEL_CACHE` из более ранней ячейки этого же кернела ещё живы — переиспользуются без повторной загрузки модели; если кернел перезапускался — эта ячейка загрузит модель заново сама.

In [ ]:
from pathlib import Path

from src.judges import build_judge
from src.run_eval import run_evaluation

SCALE_RUNS = [
    "cascade_plain_20260725",
    "cascade_s1_idk_20260725",
    "qwen2audio_plain_20260725",
    "qwen2audio_s1_idk_20260725",
]

# Переиспользуем модель, если она уже загружена в этом кернеле (ячейка "Прогон скейл-сета" выше) --
# грузим заново только если кернел перезапускался и переменных нет.
if "scale_judge" not in dir() or "MODEL_CACHE" not in dir():
    MODEL_CACHE = Path("/home/jupyter/filestore/space1/slm-audio-evidence-polina/models/qwen3-8b")
    scale_judge = build_judge(
        "local",
        model_id=str(MODEL_CACHE),
        display_name="Qwen/Qwen3-8B",
        prompt_name="judge_v1.txt",
        enable_thinking=False,
    )

for run_id in SCALE_RUNS:
    print(f"=== {run_id} ===")
    run_evaluation(
        manifest_path=TELEGRAM_MANIFEST,
        responses_path=f"results/scale_nmsqa/{run_id}/responses.jsonl",
        out_dir=f"results/scale_nmsqa/{run_id}",
        judge=scale_judge,
    )

## Коммит и пуш результатов скейл-сета

Этот клон (`/home/jupyter/project/slm-audio-evidence-polina`) склонирован напрямую с `realfork` (`PolinaSh-main/slm-audio-evidence-judge-fork`), поэтому здесь `origin` — это именно тот репозиторий, к которому привязан PR #2, `git push origin m3/llm-judge` пушит прямо туда.

`git pull` перед коммитом — на случай, если в основной (не-DataSphere) копии репозитория за это время что-то ещё запушили. Identity выставляется явно (`user.name`/`user.email`) — свежий клон в DataSphere её не наследует автоматически, а коммит без этого либо упадёт, либо уйдёт под системным именем контейнера.

Добавляются только три типа файлов на все 4 прогона (`responses_judged.jsonl`, `metrics.md`, `judge_cache.jsonl`) — конкретные пути, не `git add -A`, чтобы случайно не утащить что-то постороннее из этого DataSphere-окружения.

In [ ]:
%%bash

cd /home/jupyter/project/slm-audio-evidence-polina

git config user.name "Polina Shevyakova"
git config user.email "pasheviakova@gmail.com"

git pull origin m3/llm-judge

git add results/scale_nmsqa/*/responses_judged.jsonl \
        results/scale_nmsqa/*/metrics.md \
        results/scale_nmsqa/*/judge_cache.jsonl

echo "=== git status (staged) ==="
git status -s

git commit -m "$(cat <<'EOF'
data: score scale-set runs with frozen judge (Qwen3-8B, judge_v1.txt, no-think)

results/scale_nmsqa/{cascade,qwen2audio}_{plain,s1_idk}_20260725/ -- responses_judged.jsonl,
metrics.md, judge_cache.jsonl for all 4 scale-set runs from M2, scored in DataSphere via
scripts/judge_setup.ipynb against data/manifests/scale_nmsqa_final.jsonl (376 items).
ROLE_M4 Task 2 Step 2.
EOF
)"

git push origin m3/llm-judge

echo "=== последние коммиты ==="
git log --oneline -5